# An RL agent: policy gradients → PPO

_Capstones_

**Stack four landmark reinforcement-learning (RL) papers into one working PPO agent that solves CartPole, then LunarLander.**

---

This notebook is a practice scaffold for the **An RL agent: policy gradients → PPO** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
!pip install -q gymnasium
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — gymnasium + PyTorch (Colab)

In [ ]:
# In Colab first run:  !pip install gymnasium
# (torch is preinstalled in Colab; gymnasium is not.)
# To switch to LunarLander later, ALSO run:  !pip install "gymnasium[box2d]"
import torch
import torch.nn as nn
from torch.distributions import Categorical
import gymnasium as gym

torch.manual_seed(0)

# --- The one task switch -----------------------------------------------------
# Solve CartPole first; then set ENV_ID = "LunarLander-v2" and bump UPDATES to ~200.
# LunarLander also needs Box2D:  !pip install "gymnasium[box2d]"
ENV_ID  = "CartPole-v1"     # <- change to "LunarLander-v2" to land the rocket
SOLVED  = 475.0             # CartPole-v1 solved threshold (use ~200 for LunarLander)
UPDATES = 60                # bump to ~200 for LunarLander

EPS = 0.2                   # PPO clip range (Eq. 7)


# --- 1. Policy + value networks (REINFORCE's policy + A2C's critic, Track B). ---
class ActorCritic(nn.Module):
    def __init__(self, obs_dim, n_act, hidden=64):
        super().__init__()
        self.body = nn.Sequential(nn.Linear(obs_dim, hidden), nn.Tanh(),
                                  nn.Linear(hidden, hidden), nn.Tanh())
        self.pi   = nn.Linear(hidden, n_act)   # action logits -> policy (actor)
        self.v    = nn.Linear(hidden, 1)        # state value V(s) -> critic

    def forward(self, x):
        h = self.body(x)
        return Categorical(logits=self.pi(h)), self.v(h).squeeze(-1)


# --- 2. GAE: low-variance advantages (delta_t = r + gamma V' - V; A_t = sum (gamma lam)^l delta). ---
def compute_gae(rewards, values, dones, last_v, gamma=0.99, lam=0.95):
    adv = torch.zeros(len(rewards)); gae = 0.0
    values = values + [last_v]
    for t in reversed(range(len(rewards))):
        mask  = 1.0 - dones[t]                                   # 0 if episode ended at t
        delta = rewards[t] + gamma * values[t + 1] * mask - values[t]
        gae   = delta + gamma * lam * mask * gae                 # GAE recursion
        adv[t] = gae
    returns = adv + torch.tensor(values[:-1])                    # value targets for the critic
    return adv, returns


# --- 3. PPO update: A2C actor-critic loss (value + entropy) wrapped in PPO's clip (Eq. 7/9). ---
def ppo_update(net, opt, obs, acts, old_logp, adv, returns,
               clip_eps=EPS, c1=0.5, c2=0.01, epochs=10, use_clip=True):
    adv = (adv - adv.mean()) / (adv.std() + 1e-8)                # normalize advantages
    for _ in range(epochs):                                      # reuse the batch -- safe via the clip
        dist, value = net(obs)
        new_logp = dist.log_prob(acts)
        ratio    = torch.exp(new_logp - old_logp)                # r_t = pi_theta / pi_theta_old
        if use_clip:
            unc = ratio * adv
            clp = torch.clamp(ratio, 1 - clip_eps, 1 + clip_eps) * adv
            policy_loss = -torch.min(unc, clp).mean()            # PPO clipped surrogate (Eq. 7)
        else:
            policy_loss = -(ratio * adv).mean()                  # ABLATION: raw surrogate, no clip
        value_loss = (returns - value).pow(2).mean()             # A2C value loss
        entropy    = dist.entropy().mean()                       # A2C entropy bonus
        loss = policy_loss + c1 * value_loss - c2 * entropy      # combined objective (Eq. 9)
        opt.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(net.parameters(), 0.5)          # gradient clip (separate from L^CLIP)
        opt.step()


# --- 4. Train until solved; PRINT the return rising. ---
def train(use_clip=True, updates=UPDATES, steps_per=2048):
    torch.manual_seed(0)
    env = gym.make(ENV_ID)
    net = ActorCritic(env.observation_space.shape[0], env.action_space.n)
    opt = torch.optim.Adam(net.parameters(), lr=3e-4)
    obs, _ = env.reset(seed=0)
    ep_ret, recent, history = 0.0, [], []
    for up in range(updates):
        O, Ac, LP, R, V, D = [], [], [], [], [], []
        for _ in range(steps_per):                               # collect a rollout (ON-POLICY)
            ot = torch.as_tensor(obs, dtype=torch.float32)
            with torch.no_grad():
                dist, v = net(ot)
                a  = dist.sample()
                lp = dist.log_prob(a)                            # log pi_old recorded at collection
            nobs, rew, term, trunc, _ = env.step(int(a))
            done = term or trunc
            O.append(ot); Ac.append(a); LP.append(lp); R.append(float(rew))
            V.append(float(v)); D.append(1.0 if done else 0.0)
            ep_ret += rew; obs = nobs
            if done:
                recent.append(ep_ret); ep_ret = 0.0
                obs, _ = env.reset()
        with torch.no_grad():
            last_v = float(net(torch.as_tensor(obs, dtype=torch.float32))[1])
        adv, ret = compute_gae(R, V, D, last_v)
        ppo_update(net, opt, torch.stack(O), torch.stack(Ac), torch.stack(LP),
                   adv, ret, epochs=10, use_clip=use_clip)
        avg = sum(recent[-20:]) / max(1, len(recent[-20:]))      # avg return, last 20 episodes
        history.append(avg)
        print(f"  update {up:2d}  avg return (last 20 eps): {avg:6.1f}")
        recent = recent[-20:]
        if avg >= SOLVED:
            print(f"  -> SOLVED {ENV_ID}."); break
    env.close()
    return history

print(f"Full PPO on {ENV_ID} (clipped surrogate, GAE, actor-critic + entropy):")
clip_hist = train(use_clip=True)
print("\nABLATION -- clip removed (raw r_t*A_t, same net / GAE / lr / epochs / seed):")
noclip_hist = train(use_clip=False)
print("\nClipped avg-return trajectory:", [round(h, 1) for h in clip_hist])
print("No-clip avg-return trajectory:", [round(h, 1) for h in noclip_hist])
# Clipped PPO climbs toward ~500 and stays there (SOLVED); the no-clip ablation rises
# then destabilizes. Exact numbers vary by hardware/seed -- our small run, not a paper's.
# Switch ENV_ID to "LunarLander-v2" (and !pip install "gymnasium[box2d]", updates ~200)
# to run the SAME code on the rocket-landing task.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** In the clipped objective, suppose the probability ratio is $r_t = 1.6$ and the advantage is
            $\hat{A}_t = +2$, with clip range $\epsilon = 0.2$. What value does PPO's per-sample objective
            $\min(r_t \hat{A}_t,\ \text{clip}(r_t, 1-\epsilon, 1+\epsilon)\,\hat{A}_t)$ take, and why does
            this cap the update?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Compute the unclipped term: $r_t \hat{A}_t = 1.6 \times 2 = 3.2$. — _This is the raw surrogate gain &mdash; what REINFORCE/A2C would chase, unbounded._
- Clip the ratio: $\text{clip}(1.6,\ 0.8,\ 1.2) = 1.2$, then $1.2 \times 2 = 2.4$. — _Because the advantage is positive, the ratio is pinned at the upper edge $1+\epsilon = 1.2$._
- Take the min: $\min(3.2,\ 2.4) = 2.4$. — _The min discards the larger unclipped term, so the gradient through this sample is zeroed once the
                   ratio leaves the trust region &mdash; no incentive to push the policy further on this batch._

**Answer:** $2.4$. The clip removes the gradient once $r_t$ exceeds $1+\epsilon$, which is exactly what lets
                 PPO reuse the same rollout for several epochs without the policy blowing up.

</details>

**Problem 2.** The final CODE cell runs an ablation: it trains the identical agent with the clip removed
            (raw $r_t \hat{A}_t$, same net / GAE / learning rate / epochs / seed). Predict what the return curve
            does, and name which paper's contribution the ablation deletes.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Identify the deleted piece: removing the clip deletes PPO's (2017) contribution, leaving an A2C-style
                 surrogate that is reused for several epochs per batch. — _A2C is on-policy and meant for roughly one update per rollout; reusing a batch many times without a
                   trust region lets a single high-ratio action dominate._
- Predict the curve: it rises at first, then oscillates and crashes repeatedly instead of
                 settling near 500. — _Over-large updates on reused data push some action's ratio huge, breaking the policy; it must
                   re-learn, so the average return spikes and collapses._

**Answer:** The no-clip curve rises then oscillates/crashes; it deletes PPO's clipped surrogate objective. The
                 clip is what makes multi-epoch batch reuse safe.

</details>